# Stage 10b — Label Analysis

Thin notebook: it only **imports**, **calls** `src/label_analysis.py`, and **displays**.
Label-only QC/coverage/batch-reconciliation — no embeddings touched anywhere in this notebook.

**Input:** `<taxonomy.output_dir>/labels_v1.parquet` (written by `10a_label_export.ipynb`) + the pooled wizard long-format frame (for batch-level stats).
**Output:** `coverage_table.csv`, `qc_issues.csv`, `batch_summary.csv`, `duplicate_type_counts.csv` under `<taxonomy.output_dir>/label_analysis/`.
See `docs/10b_label_analysis_report.md` for the narrative writeup these tables feed.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.config_loader import load_config
from src import label_export as lex
from src import label_analysis as lan

cfg = load_config()
out_dir = cfg['taxonomy']['output_dir'] / 'label_analysis'
out_dir.mkdir(parents=True, exist_ok=True)


## Per-batch reconciliation (Table 1 / Table 2)
Reviewed/approved/disapproved counts and duplicate-type counts, per batch file.

In [ ]:
long = lex.load_long(cfg)
batches = lan.batch_summary(long)
display(batches)
batches.to_csv(out_dir / 'batch_summary.csv', index=False)

dup_types = lan.duplicate_type_counts(long)
display(dup_types)
dup_types.to_csv(out_dir / 'duplicate_type_counts.csv', index=False)


## Canonical-table QC and coverage (Table 3)

In [ ]:
canon, _ = lex.build_canonical(cfg)

qc_issues = lan.validate_canonical(canon)
display(qc_issues)
qc_issues.to_csv(out_dir / 'qc_issues.csv', index=False)

coverage = lan.coverage_table(canon, cfg)
display(coverage.head(40))
coverage.to_csv(out_dir / 'coverage_table.csv', index=False)

n_flag = int(coverage['below_min'].sum())
print(f"classes below min_class_count={cfg['taxonomy']['min_class_count']}: {n_flag} / {len(coverage)}")
